# Performance evaluation of the operation / temperature / time extractor

Scores the trained Snowball parser (notebook 002 output) against the manually
annotated validation set. The two sides are disjoint paper sets: the seeds in
notebook 001 come from the `B*` papers, this validation set holds the `V*` papers.

## 1. Setup

In [ ]:
from pathlib import Path
from collections import Counter
from decimal import Decimal, InvalidOperation
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

VALIDATION_FILE = Path("0_Annotated_12_Years_Fixed_Sep_24_2026.xlsx")
VALIDATION_SHEET = "0_Validation_Set"

RESULTS_FILE = Path("operation_results_ALL/operation_temperature_time_final_compact.xlsx")
RESULTS_SHEET = "Sheet1"

OUTPUT_TABLES_FILE = Path("performance_evaluation_tables.xlsx")

VALIDATION_REQUIRED_COLUMNS = [
    "title", "year", "ID", "operation_type",
    "temperature_text_span", "temperature_value", "temperature_unit",
    "time_text_span", "time_value", "time_unit",
]

RESULTS_REQUIRED_COLUMNS = [
    "paper_id", "article_title", "operation_specifier_final",
    "temperature_raw_value", "temperature_raw_units",
    "time_raw_value", "time_raw_units",
]

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)

## 1.1 Validation-set composition

Operation-type frequency over the validation sheet, with the
per-year counts of the four most frequent operations. This is the same figure as
section 1.1 of the testing notebook, run on the validation annotations, so the
two sets can be read panel against panel before any score is computed.

Scope is inherited rather than re-derived: the figure counts the nine operations
the parser was trained on, the same set section 4 scores, and reports everything
else as one excluded total. Operation names are folded through the same synonym
map used for scoring. Both are restated here because section 4 defines them
later in the notebook; the values are identical.

In [ ]:
# 1.1 — Validation-set composition: operation-type frequency over every
# annotated event.

from matplotlib.ticker import MaxNLocator

# Same values as section 4, which defines them again before scoring.
OPERATION_SYNONYMS = {"agitation": "stirring", "maturation": "aging", "refluxing": "reflux"}

IN_SCOPE_OPERATIONS = {
    "calcination", "drying", "stirring", "heating",
    "aging", "sonication", "thermal hold", "hydrothermal treatment",
    "evaporation",
}

VIS_DIR = Path("full_table_visualizations")
VIS_DIR.mkdir(exist_ok=True)

TOP_N = 4

composition = pd.read_excel(VALIDATION_FILE, sheet_name=VALIDATION_SHEET, dtype=str).dropna(how="all")
composition.columns = [str(c).strip() for c in composition.columns]

# title, year and ID are written on the first row of each paper block only.
for col in ("title", "year", "ID"):
    composition[col] = composition[col].ffill()

composition["operation_type_clean"] = (
    composition["operation_type"]
    .fillna("missing")
    .astype(str)
    .str.strip()
    .replace("", "missing")
    .str.lower()
    .map(lambda name: OPERATION_SYNONYMS.get(name, name))
)

composition["year_clean"] = (
    composition["year"].fillna("").astype(str).str.extract(r"(\d{4})")[0]
)

operation_counts = (
    composition["operation_type_clean"]
    .value_counts()
    .rename_axis("operation_type")
    .reset_index(name="frequency")
)

operation_counts["percentage"] = (
    operation_counts["frequency"] / operation_counts["frequency"].sum() * 100
)

print("\nOperation type frequency in the validation sheet:")
print(operation_counts.to_string())

operation_counts.to_csv(
    VIS_DIR / "operation_type_frequency_validation_set.csv",
    index=False
)

In [ ]:
# 1.1 — (a) operation-type distribution, (b) per-year counts for the top types.
# In scope means the nine operations of IN_SCOPE_OPERATIONS, so "other (included)"
# covers only in-scope types outside the top four.

operation_counts_clean = (
    operation_counts
    .loc[operation_counts["operation_type"].ne("missing")]
    .copy()
    .sort_values("frequency", ascending=False)
)

if operation_counts_clean.empty:
    raise ValueError("No operation types available after removing 'missing'.")

in_scope_mask = operation_counts_clean["operation_type"].isin(IN_SCOPE_OPERATIONS)
in_scope_counts = operation_counts_clean[in_scope_mask].copy()
out_of_scope_counts = operation_counts_clean[~in_scope_mask].copy()

if in_scope_counts.empty:
    raise ValueError("No in-scope operation types found in the validation sheet.")

print(
    "In scope (trained on):      "
    f"{len(in_scope_counts):3d} types, "
    f"{int(in_scope_counts['frequency'].sum()):4d} events"
)
print(
    "Out of scope (not trained): "
    f"{len(out_of_scope_counts):3d} types, "
    f"{int(out_of_scope_counts['frequency'].sum()):4d} events  "
    "(excluded from this figure)"
)

top_operations = in_scope_counts.head(TOP_N).copy()
top_order = top_operations["operation_type"].tolist()

# Overall distribution: top N plus the remaining in-scope types.
overall_distribution = top_operations[["operation_type", "frequency"]].copy()

if len(in_scope_counts) > TOP_N:
    remaining_in_scope = in_scope_counts.iloc[TOP_N:]

    print(
        f"'other (included)' = {len(remaining_in_scope)} in-scope types: "
        + ", ".join(remaining_in_scope["operation_type"].astype(str))
    )

    overall_distribution = pd.concat(
        [
            overall_distribution,
            pd.DataFrame({
                "operation_type": ["other (included)"],
                "frequency": [remaining_in_scope["frequency"].sum()],
            }),
        ],
        ignore_index=True
    )

overall_distribution["percentage"] = (
    overall_distribution["frequency"]
    / overall_distribution["frequency"].sum()
    * 100
)

overall_bar_data = overall_distribution.sort_values("frequency")


# -------------------------
# Per-year counts for the top operation types
# -------------------------

by_year_rows = composition[
    composition["operation_type_clean"].isin(top_order)
    & composition["year_clean"].notna()
].copy()

by_year_rows["year_clean"] = by_year_rows["year_clean"].astype(int)

operation_by_year = (
    by_year_rows
    .groupby(["year_clean", "operation_type_clean"])
    .size()
    .reset_index(name="frequency")
    .sort_values(["year_clean", "frequency"], ascending=[True, False])
)

operation_by_year.to_csv(
    VIS_DIR / "operation_type_top4_frequency_by_year_validation_set.csv",
    index=False
)

operation_by_year_pivot = (
    operation_by_year
    .pivot(index="year_clean", columns="operation_type_clean", values="frequency")
    .reindex(columns=top_order)
    .fillna(0)
    .sort_index()
    .astype(int)
)

if operation_by_year_pivot.empty:
    raise ValueError("No valid per-year data found for the top operation types.")


# -------------------------
# Styling, kept identical to notebook 000 so the two figures match
# -------------------------

color_map = {
    "calcination": "#0072B2",
    "drying": "#E69F00",
    "stirring": "#009E73",
    "heating": "#D55E00",
    "aging": "#CC79A7",
    "other": "#999999",
    "other (included)": "#999999"
}

fallback_cmap = plt.get_cmap("tab20")

all_labels = list(dict.fromkeys(
    overall_distribution["operation_type"].tolist() + top_order
))

fallback_color_map = {
    label: fallback_cmap(i % fallback_cmap.N)
    for i, label in enumerate(all_labels)
}

def get_operation_color(label):
    return color_map.get(
        str(label).lower(),
        fallback_color_map.get(label, "#999999")
    )


with plt.rc_context({
    "font.size": 11,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 11,
    "legend.fontsize": 10
}):

    fig, (ax_total, ax_year) = plt.subplots(
        nrows=1,
        ncols=2,
        figsize=(13, 5.5),
        gridspec_kw={"width_ratios": [1.1, 2.4]},
        constrained_layout=True
    )

    # -------------------------
    # (a) Overall distribution
    # -------------------------

    ax_total.barh(
        overall_bar_data["operation_type"],
        overall_bar_data["frequency"],
        color=[
            get_operation_color(op)
            for op in overall_bar_data["operation_type"]
        ],
        edgecolor="white",
        linewidth=0.8
    )

    max_total = max(overall_bar_data["frequency"].max(), 1)
    ax_total.set_xlim(0, max_total * 1.28)

    for y_pos, (_, row) in enumerate(overall_bar_data.iterrows()):
        ax_total.text(
            row["frequency"] + max_total * 0.025,
            y_pos,
            f'{row["frequency"]:,.0f} ({row["percentage"]:.1f}%)',
            va="center",
            ha="left",
            fontsize=10
        )

    ax_total.set_xlabel("Number of events")
    ax_total.set_ylabel("")
    ax_total.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax_total.grid(axis="x", linestyle="--", linewidth=0.6, alpha=0.35)
    ax_total.set_axisbelow(True)

    ax_total.text(
        -0.18, 1.06,
        "(a)",
        transform=ax_total.transAxes,
        fontsize=14,
        fontweight="bold",
        va="bottom",
        ha="left",
        clip_on=False
    )


    # -------------------------
    # (b) Per-year distribution
    # -------------------------

    operation_by_year_pivot.plot(
        kind="bar",
        ax=ax_year,
        width=0.82,
        color=[
            get_operation_color(op)
            for op in operation_by_year_pivot.columns
        ],
        edgecolor="white",
        linewidth=0.5
    )

    ax_year.set_xlabel("Year")
    ax_year.set_ylabel("Number of events")
    ax_year.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax_year.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.35)
    ax_year.set_axisbelow(True)

    for tick in ax_year.get_xticklabels():
        tick.set_rotation(45)
        tick.set_ha("right")

    ax_year.legend(
        title=None,
        frameon=False,
        ncol=min(len(operation_by_year_pivot.columns), 4),
        loc="upper center",
        bbox_to_anchor=(0.5, 1.16)
    )

    ax_year.text(
        -0.08, 1.06,
        "(b)",
        transform=ax_year.transAxes,
        fontsize=14,
        fontweight="bold",
        va="bottom",
        ha="left",
        clip_on=False
    )

    for ax in (ax_total, ax_year):
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    plt.savefig(
        VIS_DIR / "operation_type_combined_total_and_by_year_validation_set.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

## 2. Normalization

A record is keyed as **value + unit**. Numbers are canonicalized through `Decimal`
so `1.0`, `1` and `1.00` collapse to one key; units are folded onto a canonical
symbol (`degC`, `°C`, `℃` to `c`; `hrs`, `hour`, `hours` to `h`).

Three consequences worth stating in the report:

- **No unit conversion.** `60 min` and `1 h` are different keys, on both sides.
- **Ranges collapse to their lower bound** on both sides, because the number is
  read from the annotation span when no parsed value is present. `650-850 C` in
  the validation set and `650-850` from the parser both key as `650`, so a correct range
  extraction matches instead of scoring as a miss and a spurious hit at once.
- **Values with no number anywhere** (`room temperature`, `overnight`) key as
  `None` and drop out of every criterion. Seed building in notebook 001 rejected
  them, so the parser was never trained to emit them, and scoring them would
  charge the model for records it cannot produce.

In [ ]:
NULL_STRINGS = {"", "null", "none", "nan", "na", "n/a", "<na>"}
MISSING_UNIT = "<missing_unit>"
MISSING_COMPONENT = "<missing_component>"

TEMP_UNIT_MAP = {
    "c": "c", "°c": "c", "℃": "c",
    "degc": "c", "degreec": "c", "degreesc": "c",
    "k": "k", "kelvin": "k",
}

TIME_UNIT_MAP = {
    "h": "h", "hr": "h", "hrs": "h", "hour": "h", "hours": "h",
    "min": "min", "mins": "min", "minute": "min", "minutes": "min",
    "s": "s", "sec": "s", "secs": "s", "second": "s", "seconds": "s",
    "d": "d", "day": "d", "days": "d",
}


def clean_string(x):
    """Squash unicode spaces and whitespace; return None for anything null-like."""
    if x is None:
        return None
    try:
        if pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass

    s = str(x).replace("\ufeff", "")
    for ch in ("\u00a0", "\u202f", "\u2009"):
        s = s.replace(ch, " ")
    s = re.sub(r"\s+", " ", s).strip()

    return None if s.lower() in NULL_STRINGS else s


def is_missing(x):
    return clean_string(x) is None


def norm_text(x):
    s = clean_string(x)
    return None if s is None else s.lower()


def norm_number(x):
    s = clean_string(x)
    if s is None:
        return None

    for dash in ("\u2212", "\u2013", "\u2014"):
        s = s.replace(dash, "-")
    s = s.replace(",", "")

    pattern = r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?"
    match = re.fullmatch(pattern, s) or re.search(pattern, s)
    if not match:
        return None

    try:
        normalized = format(Decimal(match.group(0)).normalize(), "f")
    except InvalidOperation:
        return None

    if "." in normalized:
        normalized = normalized.rstrip("0").rstrip(".")

    return "0" if normalized == "-0" else normalized


def norm_unit(x, unit_map):
    s = clean_string(x)
    if s is None:
        return None

    s = s.lower().replace("℃", "°c").replace("º", "°").replace("◦", "°")
    s = s.replace("degrees", "degree").replace("degree", "deg")
    s = re.sub(r"[\s\.]", "", s)

    return unit_map.get(s, s)


def make_key(value, unit, unit_map, text_span=None):
    """VAL=<number>|UNIT=<canonical unit>, or None when no number is present."""
    num = norm_number(value)
    if num is None:
        num = norm_number(text_span)
    if num is None:
        return None

    return "VAL=%s|UNIT=%s" % (num, norm_unit(unit, unit_map) or MISSING_UNIT)



# Heating rates: °C/min, °C min−1, °C·min-1, oC min-1, K/min ... all read as c/min or k/min
# (c/h, k/h for a per-hour rate), on both sides alike.
RATE_UNIT_RE = re.compile(r"^(?:°|o|deg)?(?P<temp>[ck])(?:/|per)?(?P<per>min|h)(?:-1)?$")


def norm_rate_unit(x):
    s = clean_string(x)
    if s is None:
        return None

    s = s.lower().replace("℃", "°c").replace("º", "°").replace("◦", "°")
    for dash in ("\u2212", "\u2013", "\u2014", "\u207b"):
        s = s.replace(dash, "-")
    s = re.sub(r"[\s\.·⋅]", "", s)
    s = s.replace("minute", "min").replace("mins", "min").replace("hour", "h").replace("hr", "h")

    match = RATE_UNIT_RE.match(s)
    return "%s/%s" % (match.group("temp"), match.group("per")) if match else s


def make_rate_key(value, unit, text_span=None):
    num = norm_number(value)
    if num is None:
        num = norm_number(text_span)
    if num is None:
        return None

    return "VAL=%s|UNIT=%s" % (num, norm_rate_unit(unit) or MISSING_UNIT)


def readable_key(value):
    return re.sub(r"VAL=|UNIT=", "", str(value)).replace("|", " ")


def year_label(x):
    try:
        return str(int(Decimal(norm_number(x))))
    except (InvalidOperation, TypeError):
        return "MISSING_YEAR"


def standardize_missing_values(df):
    return df.apply(lambda col: col.map(lambda x: pd.NA if is_missing(x) else x))

## 3. Load and validate

In [ ]:
for path in (VALIDATION_FILE, RESULTS_FILE):
    if not path.exists():
        raise FileNotFoundError("File not found: %s" % path)

validation_raw = pd.read_excel(VALIDATION_FILE, sheet_name=VALIDATION_SHEET, dtype=object)
results_raw = pd.read_excel(RESULTS_FILE, sheet_name=RESULTS_SHEET, dtype=object)

for name, frame, required in (
    ("validation", validation_raw, VALIDATION_REQUIRED_COLUMNS),
    ("results", results_raw, RESULTS_REQUIRED_COLUMNS),
):
    missing = [c for c in required if c not in frame.columns]
    if missing:
        raise ValueError("Missing columns in %s file: %s" % (name, missing))

display(pd.DataFrame({
    "file": ["validation_set", "results"],
    "path": [str(VALIDATION_FILE), str(RESULTS_FILE)],
    "sheet": [VALIDATION_SHEET, RESULTS_SHEET],
    "rows_loaded": [len(validation_raw), len(results_raw)],
    "columns_loaded": [validation_raw.shape[1], results_raw.shape[1]],
}))

## 4. Matching keys and evaluation scope

Both sides are joined on the paper identifier, not on titles or on text.

Two normalizations are applied **identically to the validation set and to predictions**, so
neither side is advantaged:

1. `OPERATION_SYNONYMS` folds spelling and word-form variants of the *same*
   operation onto one name (agitation to stirring, maturation to aging,
   refluxing to reflux). Distinct operations are never merged: calcination and
   annealing stay separate, as do drying and heating.
2. `IN_SCOPE_OPERATIONS` restricts scoring to the nine operations the parser was
   trained on. An operation absent from the training data cannot be predicted, so
   an operation-anchored criterion would score it as wrong by construction, which
   measures the annotation vocabulary rather than the model. Rows outside the
   nine stay in the workbooks and in every figure of notebook 000, and the
   unrestricted scores are reported alongside the restricted ones in section 7.

**Inherited triggers.** An event whose trigger is annotated `null` repeats the
operation of the clause before it without a verb of its own (*calcined at 400 °C
for 2 h, and then at 750 °C for 180 min*). These events are removed from the
annotations before any criterion is computed, including the all-operations tables,
counted here, and listed in the `Excluded_Inherited_Trigger` sheet. A prediction
that could only have matched one of them is removed with it, so a system that does
recover one is not charged a false positive for it; the sheet records how many were
reached.

In [ ]:
validation = standardize_missing_values(validation_raw.dropna(how="all")).copy()
results = standardize_missing_values(results_raw.dropna(how="all")).copy()

validation["validation_excel_row"] = validation.index + 2
results["results_excel_row"] = results.index + 2

for col in ("title", "year", "ID"):
    validation[col] = validation[col].ffill()

validation["paper_key"] = validation["ID"].astype(str).str.strip()
validation["operation_key"] = validation["operation_type"].map(norm_text)
validation["year_label"] = validation["year"].map(year_label)
validation["temperature_key"] = [
    make_key(v, u, TEMP_UNIT_MAP, s)
    for v, u, s in zip(validation["temperature_value"], validation["temperature_unit"], validation["temperature_text_span"])
]
validation["time_key"] = [
    make_key(v, u, TIME_UNIT_MAP, s)
    for v, u, s in zip(validation["time_value"], validation["time_unit"], validation["time_text_span"])
]

results["paper_key"] = results["paper_id"].astype(str).str.strip()
results["operation_key"] = results["operation_specifier_final"].map(norm_text)
results["temperature_key"] = [
    make_key(v, u, TEMP_UNIT_MAP)
    for v, u in zip(results["temperature_raw_value"], results["temperature_raw_units"])
]
results["time_key"] = [
    make_key(v, u, TIME_UNIT_MAP)
    for v, u in zip(results["time_raw_value"], results["time_raw_units"])
]

# Heating rate: optional on both sides. A results file without it (an older 002
# run, or a system that reports no rate) simply scores no rate.
_no_validation = pd.Series(pd.NA, index=validation.index)
validation["ramp_key"] = [
    make_rate_key(v, u, s)
    for v, u, s in zip(
        validation.get("ramp_value", _no_validation),
        validation.get("ramp_unit", _no_validation),
        validation.get("ramp_rate_text_span", _no_validation),
    )
]
_no_results = pd.Series(pd.NA, index=results.index)
results["ramp_key"] = [
    make_rate_key(v, u)
    for v, u in zip(
        results.get("heating_rate_value", _no_results),
        results.get("heating_rate_units", _no_results),
    )
]

OPERATION_SYNONYMS = {"agitation": "stirring", "maturation": "aging", "refluxing": "reflux"}

IN_SCOPE_OPERATIONS = {
    "calcination", "drying", "stirring", "heating",
    "aging", "sonication", "thermal hold", "hydrothermal treatment",
    "evaporation",
}

for frame in (validation, results):
    frame["operation_key"] = frame["operation_key"].map(
        lambda k: OPERATION_SYNONYMS.get(k, k) if isinstance(k, str) else k
    )
    frame["operation_label"] = frame["operation_key"].fillna("MISSING_OPERATION")

YEAR_BY_PAPER = validation.drop_duplicates("paper_key").set_index("paper_key")["year_label"].to_dict()
TITLE_BY_PAPER = validation.drop_duplicates("paper_key").set_index("paper_key")["title"].to_dict()

results["year_label"] = results["paper_key"].map(YEAR_BY_PAPER).fillna("PAPER_NOT_IN_VALIDATION")

# Events whose trigger is annotated "null" repeat the operation of the clause before
# them without a verb of their own ("calcined at 400 °C for 2 h, and then at 750 °C for
# 180 min"). They leave every criterion here, including the all-operations tables, and
# are reported as a count and listed in their own sheet. Taking them out of the
# annotations alone would turn a correct extraction of one into a false positive, so a
# prediction that could only have matched an excluded event goes with it: at most one per
# excluded event, and never one that a remaining annotated event still accounts for.
INHERITED_TRIGGER_TOKEN = "null"
INHERITED_RECORD_COLUMNS = ["paper_key", "operation_key", "temperature_key", "time_key"]

# read_excel's defaults turn the literal string "null" into a missing value, which would
# make these rows indistinguishable from a blank trigger, so the column is re-read verbatim.
trigger_verbatim = pd.read_excel(VALIDATION_FILE, sheet_name=VALIDATION_SHEET, dtype=str, keep_default_na=False)
if len(trigger_verbatim) != len(validation_raw):
    raise ValueError("Verbatim re-read of %s disagrees on the row count." % VALIDATION_FILE)
if "operation_verb_text_span" in trigger_verbatim.columns:
    inherited_trigger = (
        trigger_verbatim.loc[validation.index, "operation_verb_text_span"]
        .str.strip().str.lower()
        .eq(INHERITED_TRIGGER_TOKEN)
    )
else:
    inherited_trigger = pd.Series(False, index=validation.index)

validation_inherited = validation[inherited_trigger].copy()
validation = validation[~inherited_trigger].copy()


def full_record_key(frame):
    cells = frame[INHERITED_RECORD_COLUMNS].astype(object).where(frame[INHERITED_RECORD_COLUMNS].notna(), "<none>")
    return [tuple(str(v) for v in row) for row in cells.to_numpy()]


excluded_counts = Counter(full_record_key(validation_inherited))
remaining_counts = Counter(full_record_key(validation))
prediction_keys = full_record_key(results)
prediction_counts = Counter(prediction_keys)
budget = {
    key: min(n, max(prediction_counts[key] - remaining_counts[key], 0))
    for key, n in excluded_counts.items()
}
neutralized = []
for index, key in zip(results.index, prediction_keys):
    if budget.get(key, 0) > 0:
        neutralized.append(index)
        budget[key] -= 1

results_inherited = results.loc[neutralized].copy()
results = results.drop(index=neutralized).copy()

print("Excluded, trigger annotated %r: %d annotated events (%d in the nine scored operations); "
      "%d predictions that could only have matched them removed as well" % (
          INHERITED_TRIGGER_TOKEN,
          len(validation_inherited),
          validation_inherited["operation_label"].isin(IN_SCOPE_OPERATIONS).sum(),
          len(results_inherited),
      ))
display(validation_inherited["operation_label"].value_counts().rename_axis("operation").reset_index(name="events"))

validation_all, results_all = validation.copy(), results.copy()
validation = validation[validation["operation_label"].isin(IN_SCOPE_OPERATIONS)].copy()
results = results[results["operation_label"].isin(IN_SCOPE_OPERATIONS)].copy()

display(pd.DataFrame({
    "dataset": ["validation", "results"],
    "rows_all_operations": [len(validation_all), len(results_all)],
    "rows_in_scope": [len(validation), len(results)],
    "rows_dropped": [len(validation_all) - len(validation), len(results_all) - len(results)],
}))

display(pd.DataFrame({
    "dataset": ["validation", "results"],
    "rows": [len(validation), len(results)],
    "temperature_records": [validation["temperature_key"].notna().sum(), results["temperature_key"].notna().sum()],
    "time_records": [validation["time_key"].notna().sum(), results["time_key"].notna().sum()],
    "temperature_time_pair_records": [
        validation[["temperature_key", "time_key"]].notna().all(axis=1).sum(),
        results[["temperature_key", "time_key"]].notna().all(axis=1).sum(),
    ],
    "heating_rate_records": [validation["ramp_key"].notna().sum(), results["ramp_key"].notna().sum()],
}))

## 5. Paper coverage

The parsed corpus is built from this same workbook, so every predicted paper must
be in the validation set: that is asserted rather than tabulated. Validation papers with no
predicted record at all are listed, which is expected for papers annotated as
containing no extractable synthesis event.

In [ ]:
unknown_papers = sorted(set(results_all["paper_key"]) - set(validation_all["paper_key"]))
assert not unknown_papers, "Predicted papers absent from the validation set: %s" % unknown_papers[:10]

validation_papers_without_results = (
    validation_all.loc[~validation_all["paper_key"].isin(set(results_all["paper_key"])), ["ID", "year", "title"]]
    .drop_duplicates("ID")
    .sort_values(["year", "ID"], ascending=[False, True], na_position="last")
)

print("Validation papers with no extracted record: %d" % len(validation_papers_without_results))
display(validation_papers_without_results.head(15))

## 6. Evaluation engine

Matching is **paper-level and multiset-based**. A record is a tuple of paper,
value and unit with no sentence alignment, so a value extracted from the right
paper but attached to the wrong sentence still counts as a true positive. A value
predicted more often than the validation set holds it counts the surplus as false
positives.

Each criterion is capped against the validation set independently, so the criteria are
nested but **not additive**: the scores of two criteria never sum to the score of
their union.

The first six criteria score operation, temperature and duration. The last four
add the heating rate: its extraction, its pairing with the set point and with the
operation, and the full triple with the rate required to agree as well (an event
annotated without a rate must also be predicted without one). The headline
`FINAL_CRITERION` stays the full triple, so earlier tables remain comparable.

In [ ]:
CRITERIA = {
    "Temperature extraction": {
        "components": ["paper_key", "temperature_key"],
        "required": ["temperature_key"],
    },
    "Duration extraction": {
        "components": ["paper_key", "time_key"],
        "required": ["time_key"],
    },
    "Temperature–duration pairing": {
        "components": ["paper_key", "temperature_key", "time_key"],
        "required": ["temperature_key", "time_key"],
    },
    "Operation–temperature pairing": {
        "components": ["paper_key", "operation_key", "temperature_key"],
        "required": ["temperature_key"],
    },
    "Operation–duration pairing": {
        "components": ["paper_key", "operation_key", "time_key"],
        "required": ["time_key"],
    },
    "Full triple": {
        "components": ["paper_key", "operation_key", "temperature_key", "time_key"],
        "required": ["temperature_key", "time_key"],
    },
    "Heating-rate extraction": {
        "components": ["paper_key", "ramp_key"],
        "required": ["ramp_key"],
    },
    "Temperature–heating-rate pairing": {
        "components": ["paper_key", "temperature_key", "ramp_key"],
        "required": ["temperature_key", "ramp_key"],
    },
    "Operation–heating-rate pairing": {
        "components": ["paper_key", "operation_key", "ramp_key"],
        "required": ["ramp_key"],
    },
    # The full triple, with the heating rate required to agree as well: a rate on one
    # side and none on the other is a miss, and two events without a rate agree.
    "Full record with heating rate": {
        "components": ["paper_key", "operation_key", "temperature_key", "time_key", "ramp_key"],
        "required": ["temperature_key", "time_key"],
    },
}

FINAL_CRITERION = "Full triple"

PERFORMANCE_COLUMNS = [
    "criterion", "validation_support", "predicted_count", "tp", "fp", "fn",
    "precision", "recall", "f1",
]

METRIC_FORMAT = {"precision": "{:.3f}", "recall": "{:.3f}", "f1": "{:.3f}"}

DISPLAY_NAME = {
    "operation_key": "operation", "temperature_key": "temperature",
    "time_key": "duration", "ramp_key": "heating_rate",
}


def counter_for(df, criterion_name):
    cfg = CRITERIA[criterion_name]
    subset = df[df[cfg["required"]].notna().all(axis=1)].copy()
    subset[cfg["components"]] = subset[cfg["components"]].fillna(MISSING_COMPONENT).astype(str)
    return Counter(map(tuple, subset[cfg["components"]].to_numpy()))


def evaluate_one(validation_df, pred_df, criterion_name):
    validation_counts = counter_for(validation_df, criterion_name)
    pred_counts = counter_for(pred_df, criterion_name)

    tp = sum(min(validation_counts[k], pred_counts[k]) for k in validation_counts.keys() & pred_counts.keys())
    fp = sum((pred_counts - validation_counts).values())
    fn = sum((validation_counts - pred_counts).values())

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        "criterion": criterion_name,
        "validation_support": sum(validation_counts.values()),
        "predicted_count": sum(pred_counts.values()),
        "tp": tp, "fp": fp, "fn": fn,
        "precision": precision, "recall": recall, "f1": f1,
    }


def evaluate_all(validation_df, pred_df, criteria_names=None):
    names = list(criteria_names or CRITERIA)
    return pd.DataFrame([evaluate_one(validation_df, pred_df, c) for c in names])[PERFORMANCE_COLUMNS]


def group_sort_key(label):
    s = str(label)
    return (0, -int(s)) if re.fullmatch(r"\d{4}", s) else (1, s)


def evaluate_by_group(validation_df, pred_df, group_col, criteria_names=None):
    labels = sorted(
        set(validation_df[group_col].dropna().astype(str)) | set(pred_df[group_col].dropna().astype(str)),
        key=group_sort_key,
    )
    rows = [
        dict(evaluate_one(
            validation_df[validation_df[group_col].astype(str) == label],
            pred_df[pred_df[group_col].astype(str) == label],
            criterion,
        ), **{group_col: label})
        for label in labels
        for criterion in (criteria_names or CRITERIA)
    ]
    return pd.DataFrame(rows)[[group_col] + PERFORMANCE_COLUMNS]


def paper_lookups(df):
    """paper -> set of operations / temperature keys / duration keys present."""
    ops, temps, durations, rates = {}, {}, {}, {}
    for paper, rate in zip(df["paper_key"], df["ramp_key"]):
        if pd.notna(rate):
            rates.setdefault(paper, set()).add(rate)
    for paper, op, t, d in zip(
        df["paper_key"], df["operation_label"], df["temperature_key"], df["time_key"]
    ):
        ops.setdefault(paper, set()).add(op)
        if pd.notna(t):
            temps.setdefault(paper, set()).add(t)
        if pd.notna(d):
            durations.setdefault(paper, set()).add(d)
    return {"operation": ops, "temperature": temps, "duration": durations, "heating_rate": rates}


def diagnose(paper, parts, validation_n, pred_n):
    """Machine-assigned failure category for one unmatched record."""
    if validation_n == pred_n:
        return ""
    if validation_n and pred_n:
        return "under_counted" if validation_n > pred_n else "over_counted"

    # A validation record that was not predicted is checked against what the parser did
    # produce for that paper, and vice versa.
    side = PREDICTED_LOOKUPS if validation_n else VALIDATION_LOOKUPS
    suffix = "_missed" if validation_n else "_spurious"

    operation = parts.get("operation_key")
    if operation is not None and operation not in side["operation"].get(paper, set()):
        return "operation" + suffix

    for component, name in (
        ("temperature_key", "temperature"), ("time_key", "duration"), ("ramp_key", "heating_rate"),
    ):
        value = parts.get(component)
        if value is None:
            continue
        available = side[name].get(paper, set())
        if value not in available:
            if value.split("|")[0] in {k.split("|")[0] for k in available}:
                return name + "_unit_mismatch"
            return name + suffix

    # Operation, temperature and duration each exist for this paper, but not
    # together: the components were found and associated with the wrong event.
    return "pairing_error"


def record_table(validation_df, pred_df, criterion_name):
    """One row per distinct record, with its validation and predicted multiplicity."""
    cfg = CRITERIA[criterion_name]
    validation_counts = counter_for(validation_df, criterion_name)
    pred_counts = counter_for(pred_df, criterion_name)

    rows = []
    for key in set(validation_counts) | set(pred_counts):
        parts = dict(zip(cfg["components"], key))
        paper = parts.pop("paper_key")
        validation_n, pred_n = validation_counts[key], pred_counts[key]

        row = {
            "year": YEAR_BY_PAPER.get(paper, "PAPER_NOT_IN_VALIDATION"),
            "paper_id": paper,
            "article_title": TITLE_BY_PAPER.get(paper, paper),
        }
        row.update({DISPLAY_NAME.get(c, c): readable_key(v) for c, v in parts.items()})
        row.update({
            "validation_count": validation_n,
            "predicted_count": pred_n,
            "tp": min(validation_n, pred_n),
            "false_negative": max(validation_n - pred_n, 0),
            "false_positive": max(pred_n - validation_n, 0),
            "status": (
                "matched" if validation_n == pred_n
                else "count_mismatch" if validation_n and pred_n
                else "false_negative" if validation_n
                else "false_positive"
            ),
            "failure_reason": diagnose(paper, parts, validation_n, pred_n),
        })
        rows.append(row)

    return pd.DataFrame(rows).sort_values(["year", "paper_id", "status"]).reset_index(drop=True)


VALIDATION_LOOKUPS = paper_lookups(validation)
PREDICTED_LOOKUPS = paper_lookups(results)

RECORD_KEY_COLUMNS = ["year", "paper_id", "operation", "temperature", "duration"]


def record_key(frame):
    return [tuple(str(v) for v in row) for row in frame[RECORD_KEY_COLUMNS].to_numpy()]


def prior_reporting_issues(path, sheet="Records_Final"):
    """Reporting issues annotated by hand in a previous run of this workbook.

    The column is written out empty and filled in Excel, so it has to survive a
    re-run: without this the next write would erase the manual annotation.
    """
    path = Path(path)
    if not path.exists():
        return {}
    try:
        prior = pd.read_excel(path, sheet_name=sheet, dtype=object)
    except (ValueError, OSError, KeyError):
        return {}
    if not {"reporting_issue", *RECORD_KEY_COLUMNS} <= set(prior.columns):
        return {}

    # Deliberately not clean_string: "none" is a real vocabulary value here, and
    # clean_string treats it as a null token.
    def text(x):
        if x is None:
            return ""
        try:
            if pd.isna(x):
                return ""
        except (TypeError, ValueError):
            pass
        return re.sub(r"\s+", " ", str(x)).strip()

    return {
        key: text(issue)
        for key, issue in zip(record_key(prior), prior["reporting_issue"])
        if text(issue)
    }

## 7. Overall performance

`overall_perf` is restricted to the nine trained operations. `overall_perf_all_operations`
repeats the same criteria over every annotated operation, for comparison.
`scope_comparison` puts precision, recall and F1 for the top four operations, the
nine trained operations and all operations side by side, one row per criterion.

In [ ]:
overall_perf = evaluate_all(validation, results)
overall_perf_all_operations = evaluate_all(validation_all, results_all)

display(overall_perf.style.format(METRIC_FORMAT))
display(overall_perf_all_operations.style.format(METRIC_FORMAT))

In [ ]:
# Section 7 by scope: the same criteria on three nested populations, side by side.
# "Top 4" is the four most frequent operations in the training set; the same four
# lead the validation and testing sets, in the same order. "Top 9" is overall_perf and
# "All operations" is overall_perf_all_operations, so the three tables cannot disagree.

TOP_OPERATIONS = {"calcination", "drying", "stirring", "heating"}

scope_perf = {
    "Top 4": evaluate_all(
        validation_all[validation_all["operation_label"].isin(TOP_OPERATIONS)],
        results_all[results_all["operation_label"].isin(TOP_OPERATIONS)],
    ),
    "Top 9": overall_perf,
    "All operations": overall_perf_all_operations,
}

scope_comparison = pd.DataFrame({"criterion": list(CRITERIA)})

for scope, perf in scope_perf.items():
    by_criterion = perf.set_index("criterion")
    for metric, short in (("precision", "P"), ("recall", "R"), ("f1", "F1")):
        scope_comparison["%s %s" % (scope, short)] = scope_comparison["criterion"].map(by_criterion[metric])

display(scope_comparison.style.format({c: "{:.3f}" for c in scope_comparison.columns if c != "criterion"}))


## 8. Performance by year

The `All years` row is the section 7 table on the same row shape, so this table
stands on its own in the report. It is excluded from the figures, which filter on
a four-digit label.

In [ ]:
year_perf = pd.concat(
    [
        overall_perf.assign(year_label="All years")[["year_label"] + PERFORMANCE_COLUMNS],
        evaluate_by_group(validation, results, "year_label"),
    ],
    ignore_index=True,
)

year_f1_pivot = (
    year_perf
    .pivot(index="year_label", columns="criterion", values="f1")
    .reindex(sorted(year_perf["year_label"].unique(), key=group_sort_key))
    .reset_index()
)

display(year_perf.style.format(METRIC_FORMAT))
display(year_f1_pivot.style.format({c: "{:.3f}" for c in year_f1_pivot.columns if c != "year_label"}))

## 9. Performance by operation

Pooled over all twelve years, for the four operation-anchored criteria.

In [ ]:
OPERATION_CRITERIA = [
    "Operation–temperature pairing",
    "Operation–duration pairing",
    FINAL_CRITERION,
    "Operation–heating-rate pairing",
]

operation_perf = evaluate_by_group(validation, results, "operation_label", OPERATION_CRITERIA)

display(operation_perf.style.format(METRIC_FORMAT))

## 10. Record-level results for the final criterion

One row per distinct operation / temperature / duration record, carrying its validation
and predicted multiplicity. The false-negative and false-positive tables are
filtered views of it, so the three can never disagree. All three go to Excel;
only the head is shown here.

Each unmatched record carries a machine-assigned `failure_reason`, decided from
what the parser produced elsewhere in the *same paper*:

| `failure_reason` | What it means |
|---|---|
| `operation_missed` / `operation_spurious` | The operation trigger was never found in that paper, or was invented there. A trigger-vocabulary problem. |
| `temperature_missed` / `duration_missed` | The value itself was never extracted anywhere in that paper. A recognition problem. |
| `temperature_spurious` / `duration_spurious` | A value was extracted that the annotation does not contain at all. |
| `temperature_unit_mismatch` / `duration_unit_mismatch` | The right number was found with the wrong unit, or with no unit. |
| `pairing_error` | Operation, temperature and duration were each found in that paper, but not bound together. The association step failed, not the recognition step. |
| `under_counted` / `over_counted` | The record is right but its multiplicity is wrong, typically the same conditions repeated across samples. |

`reporting_issue` is written out empty, as a controlled-vocabulary column to fill
by hand from the source paragraph. Suggested values are in section 14.

In [ ]:
final_records = record_table(validation, results, FINAL_CRITERION)

carried_over = prior_reporting_issues(OUTPUT_TABLES_FILE)
final_records["reporting_issue"] = [carried_over.get(k, "") for k in record_key(final_records)]

false_negatives = final_records[final_records["false_negative"] > 0].reset_index(drop=True)
false_positives = final_records[final_records["false_positive"] > 0].reset_index(drop=True)

failure_summary = (
    final_records[final_records["status"] != "matched"]
    .groupby("failure_reason")
    .agg(records=("failure_reason", "size"),
         false_negatives=("false_negative", "sum"),
         false_positives=("false_positive", "sum"))
    .sort_values("records", ascending=False)
    .reset_index()
)

print("records: %d   false negatives: %d   false positives: %d"
      % (len(final_records), len(false_negatives), len(false_positives)))
print("reporting_issue annotated by hand: %d of %d unmatched records"
      % ((final_records["reporting_issue"] != "").sum(),
         (final_records["status"] != "matched").sum()))

display(final_records["status"].value_counts().rename_axis("status").reset_index(name="records"))
display(failure_summary)
display(final_records.head(15))

# The events taken out in section 4, one row each. A Records_Final key that existed only
# because of one of them no longer appears above, so any reporting_issue written against
# it by hand is carried over here instead of being lost on this write.
def _readable_or_blank(key):
    return readable_key(key) if isinstance(key, str) else ""

reached_counts = Counter(full_record_key(results_inherited))
inherited_records = pd.DataFrame({
    "validation_excel_row": validation_inherited["validation_excel_row"],
    "year": validation_inherited["paper_key"].map(YEAR_BY_PAPER),
    "paper_id": validation_inherited["paper_key"],
    "operation": validation_inherited["operation_label"],
    "temperature": validation_inherited["temperature_key"].map(_readable_or_blank),
    "duration": validation_inherited["time_key"].map(_readable_or_blank),
    "event_text_span": validation_inherited["event_text_span"] if "event_text_span" in validation_inherited else "",
    "in_scope": validation_inherited["operation_label"].isin(IN_SCOPE_OPERATIONS),
    "reached_by_parser": [reached_counts[key] > 0 for key in full_record_key(validation_inherited)],
}).reset_index(drop=True)
inherited_records["reporting_issue"] = [
    carried_over.get(key, "") if t and d else ""
    for key, t, d in zip(record_key(inherited_records), inherited_records["temperature"], inherited_records["duration"])
]

print("Events excluded as inherited triggers: %d, of which reached by the parser anyway: %d"
      % (len(inherited_records), int(inherited_records["reached_by_parser"].sum())))

## 11. Failure cross-tabulation

`failure_reason` against `reporting_issue`: what the code did, against what the
paper did. This is the table the discussion section argues from. It stays in the
`(not yet annotated)` column until the reporting issues are filled in by hand in
`Records_Final`; re-running this notebook preserves anything already written
there.

The row that carries the argument is `no_issue`, the count of failures the prose
does not excuse.

In [ ]:
unmatched = final_records[final_records["status"] != "matched"].copy()
unmatched["reporting_issue"] = unmatched["reporting_issue"].replace("", "(not yet annotated)")

failure_cross_tab = pd.crosstab(
    unmatched["failure_reason"],
    unmatched["reporting_issue"],
    margins=True,
    margins_name="All",
).sort_values("All", ascending=False)

display(failure_cross_tab)

## 12. Figures

In [ ]:
def bar_scores(df, x_col, title, x_label, figsize=(12, 6)):
    ax = df.plot(kind="bar", x=x_col, y=["precision", "recall", "f1"], figsize=figsize, rot=45)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Score")
    ax.set_xlabel(x_label)
    ax.set_title(title)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()


operation_final_perf = (
    operation_perf[(operation_perf["criterion"] == FINAL_CRITERION) & (operation_perf["validation_support"] > 0)]
    .sort_values("validation_support", ascending=False)
)

bar_scores(
    operation_final_perf, "operation_label",
    "Final operation-temperature-time performance by operation", "Operation",
)

bar_scores(
    overall_perf, "criterion",
    "Precision, recall and F1 by criterion (all years, nine trained operations)",
    "Criterion", figsize=(11, 6),
)

In [ ]:
heatmap_pivot = (
    year_perf[year_perf["year_label"].astype(str).str.fullmatch(r"\d{4}")]
    .pivot(index="year_label", columns="criterion", values="f1")
    .sort_index()
)

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(heatmap_pivot.values, aspect="auto", vmin=0, vmax=1)

ax.set_xticks(np.arange(len(heatmap_pivot.columns)))
ax.set_yticks(np.arange(len(heatmap_pivot.index)))
ax.set_xticklabels(heatmap_pivot.columns, rotation=45, ha="right")
ax.set_yticklabels(heatmap_pivot.index)
ax.set_title("F1 score by year and criterion")

for i in range(heatmap_pivot.shape[0]):
    for j in range(heatmap_pivot.shape[1]):
        ax.text(j, i, "%.2f" % heatmap_pivot.iloc[i, j], ha="center", va="center", fontsize=8)

fig.colorbar(im, ax=ax, label="F1")
plt.tight_layout()
plt.show()

## 13. Save tables

Writing to a workbook that is open in Excel fails on Windows, sometimes as
`PermissionError` and sometimes as `OSError` errno 22. The write falls back to a
timestamped file in the same folder and says so, instead of raising.

In [ ]:
def save_sheets(path, sheets):
    path = Path(path)

    def _write(target):
        with pd.ExcelWriter(target) as writer:
            for sheet_name, frame in sheets.items():
                frame.to_excel(writer, sheet_name=sheet_name, index=False)

    try:
        _write(path)
        return path
    except (OSError, PermissionError) as error:
        stamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        fallback = path.with_name("%s_%s%s" % (path.stem, stamp, path.suffix))
        _write(fallback)
        print(
            "Could not write %s (%s: %s).\n"
            "It is almost certainly open in Excel. Wrote %s instead - "
            "close the workbook and re-run this cell to refresh the original."
            % (path.name, type(error).__name__, error, fallback.name)
        )
        return fallback


written_to = save_sheets(OUTPUT_TABLES_FILE, {
    "Overall": overall_perf,
    "Overall_All_Operations": overall_perf_all_operations,
    "Scope_Comparison": scope_comparison,
    "By_Year": year_perf,
    "F1_By_Year": year_f1_pivot,
    "By_Operation": operation_perf,
    "Records_Final": final_records,
    "Excluded_Inherited_Trigger": inherited_records,
    "False_Negatives_Final": false_negatives,
    "False_Positives_Final": false_positives,
    "Failure_Reasons": failure_summary,
    "Failure_CrossTab": failure_cross_tab.reset_index(),
    "Validation_Papers_No_Records": validation_papers_without_results,
})

print("Saved: %s" % written_to.resolve())

## 14. Controlled vocabulary for `reporting_issue`

`failure_reason` says what the *code* did. This column says what the *paper* did,
and has to be read off the source paragraph by hand. Keeping the two separate is
the point: the same `pairing_error` can come from a well-written sentence the
parser mishandled, or from a sentence no reader could resolve either.

Each value belongs to a family, and each family maps to one rule of the standard
synthesis paragraph (R1–R6). Tier **C** means the prose is under-specified: a reader
also has to infer the event or its parameters. Tier **B** means the information is
complete but not in canonical form (verb, temperature, duration).

| `reporting_issue` | Family / rule | Tier | Example prose |
|---|---|---|---|
| `operation_implied_not_named` | F1 / R1 | C | "placed in an oven at 100 °C for 15 min", "transferred to a muffle furnace at 400 °C for 4 h" |
| `ambiguous_operation_term` | F1 / R1 | C | "thermal treatment", "water-bathed", "hydrothermal aging", "preheated … to dry" |
| `nominalised_operation` | F1 / R1 | B | "the calcination was carried out at …", "under agitation", "followed by calcinations" |
| `elided_operation_verb` | F1 / R1 | B | "calcined at 500 °C for 6 h, and then at 750 °C for 4 h" |
| `concurrent_operations` | F2 / R2 | C | "heated in a water bath at 80 °C under stirring for 6 h" |
| `shared_parameter_list` | F2 / R2 | C | "followed by drying and calcination for 1 h at 900 °C" |
| `multi_stage_thermal_program` | F2 / R2 | C | "heated to 500 °C, held at this temperature for 2 h, then raised to 800 °C for 6 h" |
| `enumerated_values` | F2 / R2 | B | "calcined at 700, 800 and 900 °C for 4 h" (studied parameter written as a list) |
| `parameter_omitted` | F3 / R3 | C | "heated to 120 °C to form a viscous gel" (no duration) |
| `value_not_numeric` | F3 / R3 | C | "dried overnight", "stirred at room temperature" |
| `parameter_inherited` | F3 / R3 | C | "at the same temperature", "under the conditions above" |
| `unit_omitted_or_ambiguous` | F3 / R3 | C | "dried at 60 °C for 24" |
| `heating_rate_in_event` | F4 / R4 | B | "calcined at 600 °C (5 °C/min) for 4 h" |
| `compact_notation` | F4 / R4 | B | "650 °C/5h", "(90 °C, 2 d)", "a 6-h calcination step (550 °C)" |
| `interposed_condition` | F4 / R4 | B | "dried at 60 °C in a vacuum for 8 h" |
| `parameter_detached_from_verb` | F4 / R4 | B | "After 24 h of heating" (next sentence), "a 60 °C drying oven" |
| `nonstandard_unit_symbol` | F5 / R5 | B | "550C", "1400 o C" |
| `number_written_as_word` | F5 / R5 | B | "for three hours" |
| `synthesis_boundary_unmarked` | F6 / R6 | B | pretreatment or reduction conditions inside the synthesis paragraph |
| `parameter_outside_prose` | — | — | conditions given only in a table, figure caption or supplementary file |
| `no_issue` | — | — | the prose is canonical and the failure is the model's alone |

Rules of the standard synthesis paragraph:

- **R1**: name each step with a finite operation verb.
- **R2**: one step per clause, with its own parameters; multi-stage programmes stage by stage; state a shared operation once and list the studied parameter values at the end of the paragraph.
- **R3**: every thermal step states a numeric temperature and a numeric duration, each with a unit.
- **R4**: fixed order: verb, temperature, duration, then heating rate, atmosphere and equipment.
- **R5**: Arabic numerals and SI unit symbols, separated by a space.
- **R6**: pretreatment, reduction and characterisation conditions go outside the synthesis paragraph.

Retired values: `value_is_a_range` (ranges are scored on their lower bound and never
failed), `ramp_and_setpoint_conflated` (now `heating_rate_in_event` or
`multi_stage_thermal_program`) and `distant_or_nested_clause` (now
`interposed_condition` or `parameter_detached_from_verb`).

`no_issue` matters most for the argument. A failure marked `no_issue` is a
model limitation; everything else is evidence for the reporting standard the
discussion proposes. Write `no_issue`, never `none`: `none` is one of the null
tokens this notebook strips, and it would be read back as an empty cell.